# Relationformer per P&ID — riproduzione approssimata

Questo notebook implementa la pipeline Relationformer del paper locale 2411.13929v3.pdf. Il pretraining usa i 500 sintetici Dataset-P&ID al posto dei 2.000 originali non disponibili, quindi non garantisce gli stessi risultati numerici.

Per il fine tuning occorrono ancora **60 P&ID reali annotati**, forniti separatamente. Non usare OPEN100 per sostituirli. Dataset PID è usato nel training; **OPEN100 e PID2Graph Synthetic rimangono benchmark indipendenti.**

Abilita una GPU nelle impostazioni Kaggle (si usa una GPU anche se ne sono disponibili due). Abilita Internet per la prima installazione e i pesi ImageNet. Carica il codice aggiornato e PID2Graph come Kaggle Dataset, includendo models/ops; non serve compilare CUDA. Esegui le celle dall'alto verso il basso dopo aver impostato i percorsi.

I default pubblicati sono: input 512×512, batch effettivo 20, 80 epoche, LR 1e-4 / backbone 3e-5, ResNet-101, 400 object token + 1 relation token e loss 2/2/1/4/3. AdamW, ImageNet, scheduler, patience e soglie sono scelte implementative documentate nel README; non tutti sono specificati dal paper.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess
from IPython.display import display, FileLink

# Se hai caricato più copie della repo, assegna REPO_SOURCE manualmente.
candidates = [p.parent for p in Path("/kaggle/input").rglob("train.py")
              if (p.parent / "models/relationformer_2D.py").is_file()]
assert len(candidates) == 1, f"Imposta REPO_SOURCE con il percorso della repo: {candidates}"
REPO_SOURCE = candidates[0]
REPO = Path("/kaggle/working/relationformer")
if not (REPO / "train.py").is_file():
    shutil.copytree(REPO_SOURCE, REPO,
        ignore=shutil.ignore_patterns(".git", ".venv", "__pycache__", "PID2Graph", "PID2Graph.zip", "data", "trained_weights"))
os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["MPLCONFIGDIR"] = "/kaggle/working/matplotlib"
os.environ["RELATIONFORMER_CACHE_DIR"] = "/kaggle/working/pid-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch
assert torch.cuda.is_available(), "Abilita una GPU nelle impostazioni del notebook"
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0), "memoria libera/totale:", torch.cuda.mem_get_info(0))
print("Spazio disco:", shutil.disk_usage("/kaggle/working"))
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True)


## Configurare i dati e il checkpoint

Occorrono il checkpoint di pretraining, `PID2Graph/Patched/Dataset PID` e 60 disegni reali annotati, già patchati. Per patchare i reali completi usa scripts/patch_pid.py --input-dir ... --output-dir .../Real. Il notebook si ferma esplicitamente se mancano i dati reali; OPEN100 non può sostituirli.

Lo split per disegno e il seed devono restare uguali al pretraining. I reali hanno tre augmentation per epoca rispetto a una per patch sintetica; la percentuale finale dipende dalle patch disponibili.


In [ ]:
PRETRAINED = Path("/kaggle/input/your-pretraining-run/best.pt")  # oppure last.pt se appropriato
REAL_PATCHES = Path("/kaggle/input/your-real-pids/Patched/Real")
BENCHMARK = Path("/kaggle/input/your-pid2graph/PID2Graph")  # contiene Patched e Complete
SYNTHETIC_PATCHES = BENCHMARK / "Patched/Dataset PID"
assert PRETRAINED.is_file(), "Imposta il checkpoint del pretraining"
assert SYNTHETIC_PATCHES.is_dir(), "Manca Patched/Dataset PID"
assert REAL_PATCHES.is_dir(), "Mancano i 60 P&ID reali annotati; non usare OPEN100"
real_drawings = [p for p in REAL_PATCHES.iterdir() if p.is_dir() and any(p.glob("*.graphml"))]
synthetic_drawings = [p for p in SYNTHETIC_PATCHES.iterdir() if p.is_dir() and any(p.glob("*.graphml"))]
assert len(real_drawings) == 60, f"Attesi 60 disegni reali: trovati {len(real_drawings)}"
assert len(synthetic_drawings) >= 500, "Occorrono almeno 500 disegni sintetici"
TRAIN_ROOT = REPO / "data/finetune/Patched"
TRAIN_ROOT.mkdir(parents=True, exist_ok=True)
for name, source in [("Dataset PID", SYNTHETIC_PATCHES), ("Real", REAL_PATCHES)]:
    destination = TRAIN_ROOT / name
    if not destination.exists(): destination.symlink_to(source, target_is_directory=True)


In [ ]:
import yaml
config = yaml.safe_load((REPO / "configs/road_2D.yaml").read_text())
config["DATA"]["BATCH_SIZE"] = 2  # se OOM: 1; batch effettivo rimane 20
config["DATA"]["NUM_WORKERS"] = 2
config["DATA"]["CACHE_IMAGES"] = False  # evita memmap >20 GB: lettura lazy
config["TRAIN"]["EFFECTIVE_BATCH_SIZE"] = 20
config["TRAIN"]["MAX_HOURS"] = 9.5  # budget di questa sessione, non completamento del training
config["TRAIN"]["EARLY_STOPPING_PATIENCE"] = 10  # assunzione, attiva solo in fine tuning
CONFIG = REPO / "kaggle_config.yaml"
CONFIG.write_text(yaml.safe_dump(config))
config["TRAIN"]["REAL_WEIGHT"] = 3
CONFIG.write_text(yaml.safe_dump(config))


## Fine tuning

--pretrained carica solo i pesi e azzera ottimizzatore, scheduler ed early stopping. --resume riprende invece tutto il run. Il miglior checkpoint minimizza la validation loss. La patience di 10 validation è una scelta esplicita, non un parametro pubblicato dagli autori.


In [ ]:
OUT = Path("/kaggle/working/finetune")
RESUME = None  # last.pt del fine tuning, non del pretraining
if RESUME is not None and not OUT.exists():
    shutil.copytree(RESUME.parent, OUT)
    RESUME = OUT / RESUME.name
command = ["train.py", "--phase", "finetune", "--config", CONFIG, "--data-root", TRAIN_ROOT,
           "--output-dir", OUT, "--device", "cuda",
           "--synthetic-source", "Dataset PID",
           "--real-drawings", 60, "--finetune-synthetic-drawings", 500]
command += ["--resume", RESUME] if RESUME is not None else ["--pretrained", PRETRAINED]
run(*command)


In [ ]:
import matplotlib.pyplot as plt
history_path = OUT / "history.jsonl"
if history_path.exists():
    rows = [json.loads(line) for line in history_path.read_text().splitlines() if line]
    plt.plot([r["epoch"] for r in rows], [r["train_loss"]["total"] for r in rows], label="train")
    plt.plot([r["epoch"] for r in rows], [r["validation"]["loss"]["total"] for r in rows], label="validation")
    plt.xlabel("Epoca"); plt.ylabel("Loss totale"); plt.legend(); plt.show()
    previews = sorted((OUT / "validation").glob("*/prediction.png"))
    if previews:
        from PIL import Image
        display(Image.open(previews[-1]))
        display(Image.open(previews[-1].with_name("truth.png")))
else:
    print("Nessuna epoca completa: usa last.pt per continuare; la validation viene eseguita a fine epoca.")


## Valutazione indipendente e risultati qualitativi

Le metriche sono symbol mAP@0.5, node AP@0.5 ignorando la classe, edge mAP con Hungarian/gIoU. I border sono esclusi dalla node AP di default; per gli archi delle patch sono mantenuti. Il protocollo A1 non specifica tutti i dettagli dell'integrazione AP: consulta README.

La modalità patched usa le patch pubbliche. Stitched ripatcha Complete con la geometria nota (1500/750) e fonde le predizioni nelle coordinate originali: non usa i vecchi array pickle degli offset. La dimensione delle patch Synthetic fornite è 2000, quindi i due protocolli non hanno geometria identica. Non tarare soglie o hyperparametri sui benchmark.

L'evaluation completa è costosa: impostala True a training concluso, anche in una sessione separata. MAX_SAMPLES=1 è solo uno smoke test e non misura le prestazioni del benchmark.


In [ ]:
RUN_EVALUATION = False
MAX_SAMPLES = None  # 1 per prova veloce; None per tutti i benchmark
if RUN_EVALUATION:
    assert (OUT / "best.pt").is_file(), "Attendi almeno una validation completa"
    assert (BENCHMARK / "Complete").is_dir() and (BENCHMARK / "Patched").is_dir()
    EVAL_OUT = Path("/kaggle/working/evaluation")
    command = ["evaluate_pid.py", "--dataset-root", BENCHMARK, "--checkpoint", OUT / "best.pt",
               "--output-dir", EVAL_OUT, "--mode", "both", "--batch-size", 2,
               "--sources", "PID2Graph OPEN100", "PID2Graph Synthetic"]
    if MAX_SAMPLES is not None: command += ["--max-samples", MAX_SAMPLES]
    run(*command)
    print((EVAL_OUT / "metrics.json").read_text())
    from PIL import Image
    examples = sorted(EVAL_OUT.rglob("prediction.png"))
    if examples: display(Image.open(examples[0]))


## Esportazione

Ogni disegno completo valutato produce graph.json, graph.graphml e prediction.png, oltre alle predizioni delle patch e al manifest. Per altri P&ID usa predict_image.py IMAGE --checkpoint best.pt --output-dir OUTPUT; aggiungi --single-patch per una patch già ritagliata.


In [ ]:
# Save Version / Run All conserva /kaggle/working negli Output del notebook.
# Puoi scaricare i singoli file da Output, oppure questo archivio del run.
if (OUT / "last.pt").is_file():
    archive = shutil.make_archive(str(OUT) + "_run", "zip", OUT)
    display(FileLink(archive))
    print("Conserva checkpoint, config.yaml e split.json; riusa gli stessi dati.")
